In [1]:
import numpy as np

from pathlib import Path

KITTI_CARLA_PATH = "/home/ksas/Public/datasets/KITTI-CARLA/dataset"
MAP = "Town01"

In [2]:
# Define PLY types
ply_dtypes = dict([
    (b'char', 'i1'),
    (b'int8', 'i1'),
    (b'uchar', 'b1'),
    (b'uchar', 'u1'),
    (b'uint8', 'u1'),
    (b'short', 'i2'),
    (b'int16', 'i2'),
    (b'ushort', 'u2'),
    (b'uint16', 'u2'),
    (b'int', 'i4'),
    (b'int32', 'i4'),
    (b'uint', 'u4'),
    (b'uint32', 'u4'),
    (b'float', 'f4'),
    (b'float32', 'f4'),
    (b'double', 'f8'),
    (b'float64', 'f8')
])


# Numpy reader format
valid_formats = {'ascii': '', 'binary_big_endian': '>', 'binary_little_endian': '<'}

def parse_header(plyfile, ext):
    # Variables
    line = []
    properties = []
    num_points = None

    while b'end_header' not in line and line != b'':
        line = plyfile.readline()
    
        if b'element' in line:
            line = line.split()
            num_points = int(line[2])

        elif b'property' in line:
            line = line.split()
            properties.append((line[2].decode(), ext + ply_dtypes[line[1]]))

    return num_points, properties

def read_ply(filename):
    """
    Read ".ply" files

    Parameters
    ----------
    filename : string
        the name of the file to read.

    Returns
    -------
    result : array
        data stored in the file

    Examples
    --------
    Store data in file

    >>> points = np.random.rand(5, 3)
    >>> values = np.random.randint(2, size=10)
    >>> write_ply('example.ply', [points, values], ['x', 'y', 'z', 'values'])

    Read the file

    >>> data = read_ply('example.ply')
    >>> values = data['values']
    array([0, 0, 1, 1, 0])
    
    >>> points = np.vstack((data['x'], data['y'], data['z'])).T
    array([[ 0.466    0.595    0.324]
             [ 0.538    0.407    0.654]
             [ 0.850    0.018    0.988]
             [ 0.395    0.394    0.363]
             [ 0.873    0.996    0.092]])

    """

    with open(filename, 'rb') as plyfile:
        # Check if the file start with ply
        if b'ply' not in plyfile.readline():
            raise ValueError('The file does not start whith the word ply')

        # get binary_little/big or ascii
        fmt = plyfile.readline().split()[1].decode()
        if fmt == "ascii":
            raise ValueError('The file is not binary')

        # get extension for building the numpy dtypes
        ext = valid_formats[fmt]

        # Parse header
        num_points, properties = parse_header(plyfile, ext)

        # Get data
        data = np.fromfile(plyfile, dtype=properties, count=num_points)

    return data

In [33]:
frame_0000 = read_ply("/home/ksas/Public/datasets/KITTI-CARLA/dataset/Town01/generated/frames/frame_0000.ply")
pose_liadr = read_ply("/home/ksas/Public/datasets/KITTI-CARLA/dataset/Town01/generated/poses_lidar.ply")

In [35]:
print(frame_0000.shape)
print(frame_0000.dtype)
print(frame_0000[:][["x", "y", "z", "cos_angle_lidar_surface", "instance"]])
new_frame_0000 = np.stack([frame_0000["x"], frame_0000["y"], frame_0000["z"], frame_0000["cos_angle_lidar_surface"]], axis=1)
print(new_frame_0000)

(131337,)
[('x', '<f4'), ('y', '<f4'), ('z', '<f4'), ('cos_angle_lidar_surface', '<f4'), ('timestamp', '<f4'), ('instance', '<u4'), ('semantic', '<u4')]
[(-62.462616  ,  3.9062499e-05, -0.13843842, 0.8280017 , 307)
 (-62.468044  ,  1.7845702e-01, -0.13845092, 0.82792616, 307)
 (-62.473495  ,  3.5687500e-01, -0.13846466, 0.82784396, 307) ...
 ( -0.7201953 , -6.1914059e-03, -0.33279052, 0.41141778, 259)
 ( -0.72017574, -4.1210935e-03, -0.3327696 , 0.41143864, 259)
 ( -0.7201367 , -2.0703124e-03, -0.3327487 , 0.41146576, 259)]
[[-6.2462616e+01  3.9062499e-05 -1.3843842e-01  8.2800168e-01]
 [-6.2468044e+01  1.7845702e-01 -1.3845092e-01  8.2792616e-01]
 [-6.2473495e+01  3.5687500e-01 -1.3846466e-01  8.2784396e-01]
 ...
 [-7.2019529e-01 -6.1914059e-03 -3.3279052e-01  4.1141778e-01]
 [-7.2017574e-01 -4.1210935e-03 -3.3276960e-01  4.1143864e-01]
 [-7.2013670e-01 -2.0703124e-03 -3.3274871e-01  4.1146576e-01]]


array([(   0.      ,   0.     , 0.        , 0.0000000e+00),
       (   0.      ,   0.     , 0.        , 1.0000000e-03),
       (   0.      ,   0.     , 0.        , 2.0000001e-03), ...,
       (-117.028336, 200.80695, 0.00154448, 4.9999701e+02),
       (-117.028336, 200.80695, 0.00154448, 4.9999802e+02),
       (-117.028336, 200.80695, 0.00154448, 4.9999902e+02)],
      dtype=[('x', '<f4'), ('y', '<f4'), ('z', '<f4'), ('timestamp', '<f4')])